# Qwen3.5-9B QLoRA: grading-metric checkpoint selection

- Text-only QLoRA with an 80/10/10 split by unique question (seed 42).
- Inputs: question, desired answer, student answer, and ordered criteria.
- Target: one JSON array of binary criterion scores, e.g. `[1, 0, 1]`.
- Criterion maxima and student scores are evaluation metadata only.
- Generate grades for the full validation set about four times per epoch.
- Select and restore the highest-validation-exact-array-accuracy checkpoint; use the same metric for early stopping and LR plateaus.
- Log validation loss as a diagnostic; the test set is used only after training.
- Thinking is disabled for training and generation.
- Run setup and data checks first; set `RUN_TRAINING=True` to load/train the model.
- Training creates new text-only adapters; use a new output directory for each experiment.

## Install dependencies

- Run once in the Linux GPU training environment; restart the kernel afterward.
- All package installs use pip, including `ipywidgets` for notebook progress bars.
- CUDA 13.0 compiler and headers use [NVIDIA pip wheels](https://pypi.org/project/cuda-toolkit/13.0.0/).
- System GCC/G++ must already be available for Triton JIT and CUDA extension compilation; pip cannot install the system compiler. On Ubuntu, install them with `sudo apt-get install build-essential` if missing.
- Optional DeltaNet kernels may need compilation; failed installs leave the PyTorch fallback.
- API references: [Qwen3.5](https://huggingface.co/docs/transformers/model_doc/qwen3_5), [TRL SFT](https://huggingface.co/docs/trl/sft_trainer).

In [ ]:
%pip install ipywidgets
%conda install -y -c conda-forge -c nvidia gcc_linux-64 cuda-nvcc
%pip install -U torch transformers accelerate peft datasets bitsandbytes trl sentencepiece pandas scikit-learn ninja packaging huggingface_hub python-dotenv

import os
import shutil
import subprocess
import sys
from importlib.metadata import distribution
from pathlib import Path

# Locate the pip-installed CUDA compiler instead of assuming a Conda layout.
nvcc_dist = distribution("nvidia-cuda-nvcc")
nvcc = next(
    (Path(nvcc_dist.locate_file(file)).resolve() for file in nvcc_dist.files or []
     if str(file).endswith("/bin/nvcc")),
    None,
)
if nvcc is None or not nvcc.is_file():
    raise RuntimeError("CUDA nvcc was not found; check the CUDA pip installation above.")
cuda_home = nvcc.parent.parent
os.environ["CUDA_HOME"] = str(cuda_home)
os.environ["PATH"] = str(nvcc.parent) + os.pathsep + os.environ.get("PATH", "")
os.environ["CPATH"] = str(cuda_home / "include") + os.pathsep + os.environ.get("CPATH", "")
os.environ.setdefault("TORCH_CUDA_ARCH_LIST", "8.9")  # RTX 4090
os.environ.setdefault("MAX_JOBS", "1")
if (system_cc := shutil.which("gcc")):
    os.environ["CC"] = system_cc
if (system_cxx := shutil.which("g++")):
    os.environ["CXX"] = system_cxx
if not system_cc or not system_cxx:
    print("System GCC/G++ missing; on Ubuntu run: sudo apt-get install build-essential")

for package_args in (["flash-linear-attention[cuda]"], ["causal-conv1d", "--no-build-isolation"]):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-U", *package_args], check=False,
    )
    if result.returncode:
        print(f"Optional install failed: {package_args[0]}; using the PyTorch fallback.")

## Imports and hardware

- One CUDA GPU is required for model loading and QLoRA training.
- Dataset and tokenizer checks can run on CPU.

In [ ]:
import ast
import csv
import gc
import importlib.util
import json
import math
import os
import random
import re
import shutil
from pathlib import Path

# Set the compiler before importing PyTorch/Triton.
cc = os.environ.get("CC") or next((
    path for name in (
        "gcc", "clang"
    ) if (path := shutil.which(name))
), None)
if cc:
    os.environ["CC"] = cc

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from huggingface_hub import login, upload_folder
from IPython.display import display
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, cohen_kappa_score, precision_recall_fscore_support
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, EarlyStoppingCallback, set_seed
from trl import SFTConfig, SFTTrainer
from tqdm.auto import tqdm

from dotenv import load_dotenv

load_dotenv()
hf_token = os.environ["HF_TOKEN"]

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Compiler:", cc)
for package in ("causal_conv1d", "fla"):
    print(f"{package} installed:", importlib.util.find_spec(package) is not None)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GiB")

## Configuration

- Keep paths, seeds, LoRA settings, and training controls here.
- `MAX_SEQ_LENGTH=None` uses the longest formatted training/validation example, rounded up to 32 tokens.
- An explicit cap rejects overlength training/validation rows instead of silently removing their labels.
- Batch size × accumulation gives the effective batch size on one GPU.
- `RUN_TRAINING` gates model loading, training, saving, and inference.
- `RUN_TEST_EVALUATION` adds two test-set generation passes after training.
- `RUN_HF_UPLOAD` uploads the final adapter and tokenizer only after test evaluation completes.
- Physical batch 1 limits memory for the longer current dataset; accumulation 8 retains effective batch 8.

- `CHECKPOINT_METRIC` selects a generated validation grading metric; higher is better.
- Validation generation adds one full validation pass at every evaluation, even when test evaluation is disabled.


In [ ]:
MODEL_ID = "Qwen/Qwen3.5-9B"
DATASET_PATH = "./training_data.csv"
OUTPUT_DIR = "./qwen35-9b-grading-lora-grading-metric"
RUN_TRAINING = True
RUN_TEST_EVALUATION = True
RUN_HF_UPLOAD = True  # Opt in only when ready to publish the evaluated adapter.
HF_REPO_ID = "SmuFypTeam5/GradingQlora"
evaluation_completed = False
SEED = 42
SPLIT_SEED = 42
MAX_SEQ_LENGTH = None  # Auto-size from formatted training/validation data.

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.05
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8  # Effective batch size 8 on one GPU.
VALIDATIONS_PER_EPOCH = 4
# Exact-array accuracy counts invalid predictions as incorrect.
CHECKPOINT_METRIC = "eval_grading_exact_match_accuracy"
# Alternative: "eval_grading_final_score_exact_accuracy".
if CHECKPOINT_METRIC not in {
    "eval_grading_exact_match_accuracy", "eval_grading_final_score_exact_accuracy",
}:
    raise ValueError("Choose a supported grading accuracy metric for checkpoint selection.")
LR_REDUCTION_FACTOR = 0.5
LR_PATIENCE = 1
MIN_LEARNING_RATE = 1e-6
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_THRESHOLD = 0.0
GENERATION_MIN_TOKENS = 128
GENERATION_TOKENS_PER_CRITERION = 8

set_seed(SEED)

## Load the dataset

- Require prompt fields, binary labels, criterion maxima, and final-score metadata.
- Keep blank student answers empty and preserve literal `NA` text.

In [ ]:
required_columns = [
    "question", "desired_answer", "student_answer", "criteria[]",
    "criteria_score[]", "criteria_max_score[]", "student_score",
]
df = pd.read_csv(DATASET_PATH, keep_default_na=False)
missing = sorted(set(required_columns) - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if df.empty:
    raise ValueError("The dataset is empty.")
print("Rows:", len(df))
display(df.head())

## Parse and validate rows

- Accept JSON/Python arrays and legacy quoted bracket arrays.
- Use label counts and `Answer` boundaries to preserve commas inside legacy criteria.
- Require nonempty criteria, aligned arrays, binary labels, and finite nonnegative score metadata.
- Keep recorded final scores even if they differ from the sum of criterion labels.

In [ ]:
def parse_array(value, expected_count=None):
    """Parse structured arrays or legacy bracket text without losing criterion commas."""
    items = value if isinstance(value, list) else None
    if items is None:
        if not isinstance(value, str):
            raise TypeError(f"Expected an array or string, got {type(value).__name__}.")
        text = value.strip()
        # Unwrap quoted JSON/Python strings before handling legacy unquoted entries.
        for _ in range(3):
            try:
                parsed = json.loads(text)
            except json.JSONDecodeError:
                try:
                    parsed = ast.literal_eval(text)
                except (ValueError, SyntaxError):
                    break
            if isinstance(parsed, list):
                items = parsed
                break
            if not isinstance(parsed, str):
                raise ValueError(f"Expected a list, got {value!r}.")
            text = parsed.strip()
        if items is None:
            if not (text.startswith("[") and text.endswith("]")):
                raise ValueError(f"Expected a bracketed list, got {value!r}.")
            inner = text[1:-1].strip()
            if not inner:
                items = []
            elif expected_count == 1:
                items = [inner]
            elif re.search(r",\s*(?=Answer\s)", inner):
                items = re.split(r",\s*(?=Answer\s)", inner)
            else:
                items = next(csv.reader([inner], skipinitialspace=True))
            items = [item.strip() for item in items]
    if expected_count is not None and len(items) != expected_count:
        raise ValueError(f"Expected {expected_count} entries, parsed {len(items)}: {value!r}")
    return items


def parse_binary_scores(value):
    values = parse_array(value)
    if any(type(x) not in (int, str) or str(x) not in ("0", "1") for x in values):
        raise ValueError(f"Expected binary integers, got {value!r}")
    return [int(x) for x in values]


def parse_score_metadata(value):
    values = parse_array(value)
    if any(isinstance(x, bool) for x in values):
        raise ValueError("Criterion maxima cannot be booleans.")
    return [float(x) for x in values]


records = []
errors = []
for index, row in enumerate(df.to_dict("records")):
    try:
        scores = parse_binary_scores(row["criteria_score[]"])
        criteria = parse_array(row["criteria[]"], expected_count=len(scores))
        maxima = parse_score_metadata(row["criteria_max_score[]"])
        final_score = float(row["student_score"])
        if not criteria or any(not isinstance(c, str) or not c.strip() for c in criteria):
            raise ValueError("Criteria must be nonempty strings.")
        if len(criteria) != len(maxima):
            raise ValueError("Criterion, label, and maximum-score counts must match.")
        if any(not math.isfinite(x) or x < 0 for x in [*maxima, final_score]):
            raise ValueError("Score metadata must be finite and nonnegative.")
        if not str(row["question"]).strip() or not str(row["desired_answer"]).strip():
            raise ValueError("Question and desired answer must be nonempty.")
        records.append({
            "row_index": index,
            **{key: str(row[key]) for key in ("question", "desired_answer", "student_answer")},
            "criteria": criteria, "criteria_score": scores,
            "criteria_max_score": maxima, "student_score": final_score,
        })
    except (ValueError, TypeError, SyntaxError) as error:
        errors.append(f"Row {index}: {error}")
if errors:
    raise ValueError(f"{len(errors)} invalid rows:\n" + "\n".join(errors[:20]))
print(f"All {len(records)} rows passed validation.")
score_disagreements = sum(
    not math.isclose(sum(s * m for s, m in zip(r["criteria_score"], r["criteria_max_score"])), r["student_score"])
    for r in records
)
print("Recorded final scores differing from criterion totals:", score_disagreements)

## Grading prompt

- Return only the ordered JSON array of binary decisions.
- Share one prompt builder between training and inference.

In [ ]:
SYSTEM_PROMPT = (
    "You are a strict grading assistant.\n\n"
    "Your task is to evaluate whether a student's answer fulfills each grading criterion.\n\n"
    "For each criterion:\n"
    "- Output 1 if the criterion is fulfilled.\n"
    "- Output 0 if the criterion is not fulfilled.\n\n"
    "Evaluate each criterion independently.\n\n"
    "Return ONLY a valid JSON array containing 0 or 1.\n"
    "The output array must contain exactly one value for each criterion, "
    "in the same order as the criteria provided.\n\n"
    "Do not include explanations.\n"
    "Do not include markdown.\n"
    "Do not include any text before or after the JSON array."
)

print(SYSTEM_PROMPT)

## Build examples

- Retain structured fields for evaluation; do not parse fields back out of prompt text.
- Keep the assistant completion separate from the prompt.

In [ ]:
def make_prompt(question, desired_answer, student_answer, criteria):
    criteria_text = "\n".join(f"{i}. {text}" for i, text in enumerate(criteria, 1))
    user_text = (
        f"Question:\n{question}\n\n"
        f"Desired answer:\n{desired_answer}\n\n"
        f"Student answer:\n{student_answer}\n\n"
        f"Criteria:\n{criteria_text}"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]


dataset = Dataset.from_list([
    {
        **row,
        "prompt": make_prompt(row["question"], row["desired_answer"], row["student_answer"], row["criteria"]),
        "completion": [{"role": "assistant", "content": json.dumps(row["criteria_score"])}],
    }
    for row in records
])
print(dataset[0]["prompt"])

## Split by question

- Shuffle unique stripped question strings with `SPLIT_SEED`.
- Allocate 80/10/10% of questions; row proportions can differ.
- Require nonempty, disjoint train, validation, and test sets.

In [ ]:
question_keys = [row["question"].strip() for row in records]
unique_questions = list(dict.fromkeys(question_keys))
random.Random(SPLIT_SEED).shuffle(unique_questions)
n_questions = len(unique_questions)
train_end = int(n_questions * 0.80)
validation_end = train_end + int(n_questions * 0.10)
question_splits = {
    "train": set(unique_questions[:train_end]),
    "validation": set(unique_questions[train_end:validation_end]),
    "test": set(unique_questions[validation_end:]),
}
if any(not questions for questions in question_splits.values()):
    raise ValueError("Too few unique questions for nonempty 80/10/10 splits.")
assert sum(map(len, question_splits.values())) == len(set.union(*question_splits.values()))
question_to_split = {q: name for name, questions in question_splits.items() for q in questions}
split_indices = {name: [] for name in question_splits}
for index, question in enumerate(question_keys):
    split_indices[question_to_split[question]].append(index)
splits = {name: dataset.select(indices) for name, indices in split_indices.items()}
train_dataset, validation_dataset, test_dataset = (splits[name] for name in ("train", "validation", "test"))
for name, split in splits.items():
    print(f"{name}: {len(split)} rows / {len(question_splits[name])} questions")

## Tokenizer

- Use the text tokenizer for both supervised examples and inference.
- Preserve the same chat template and disabled thinking mode throughout.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer:", tokenizer.__class__.__name__)

## Tokenize and verify completion masks

- Pre-tokenize training/validation examples; pass only token IDs and labels to TRL.
- Verify the generation prompt is an exact prefix of the full conversation.
- Mask prompt labels with `-100`; supervise the complete assistant turn, including its end token.
- Measure actual lengths before model loading; never silently truncate completions.

In [ ]:
def tokenize_example(example):
    prompt_ids = tokenizer.apply_chat_template(
        example["prompt"], tokenize=True, return_dict=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    input_ids = tokenizer.apply_chat_template(
        example["prompt"] + example["completion"], tokenize=True, return_dict=False,
        add_generation_prompt=False, enable_thinking=False,
    )
    boundary = len(prompt_ids)
    if input_ids[:boundary] != prompt_ids or boundary >= len(input_ids):
        raise ValueError(f"Invalid prompt/completion boundary at row {example['row_index']}.")
    return {"input_ids": input_ids, "labels": [-100] * boundary + input_ids[boundary:]}


formatted_train, formatted_validation = (
    split.map(tokenize_example, remove_columns=split.column_names, desc=f"Tokenize {name}")
    for name, split in (("train", train_dataset), ("validation", validation_dataset))
)
lengths = [len(ids) for split in (formatted_train, formatted_validation) for ids in split["input_ids"]]
measured_max_length = max(lengths)
sequence_limit = MAX_SEQ_LENGTH if MAX_SEQ_LENGTH is not None else math.ceil(measured_max_length / 32) * 32
if sequence_limit < measured_max_length:
    raise ValueError(
        f"{sum(length > sequence_limit for length in lengths)} examples exceed {sequence_limit} tokens; "
        f"longest is {measured_max_length}. Increase MAX_SEQ_LENGTH or use None."
    )
print("Training/validation token lengths:", pd.Series(lengths).describe().round(1).to_dict())
print("Sequence limit:", sequence_limit)
print("Example supervised completion:", tokenizer.decode(
    [label for label in formatted_train[0]["labels"] if label != -100], skip_special_tokens=False,
))

## Quantisation and model loading

- Load the text backbone once with NF4, double quantisation, and BF16/FP16 compute.
- SDPA handles full attention; optional DeltaNet kernels handle linear attention.
- Keep the full model on one GPU; CPU/disk dispatch is not used for training.

In [ ]:
if RUN_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("QLoRA training requires a CUDA GPU.")
    if not cc:
        raise RuntimeError("Install a C compiler with the dependency cell, then restart the kernel.")
    if int(os.environ.get("WORLD_SIZE", "1")) != 1:
        raise RuntimeError("This notebook's batch and validation schedule assume one training GPU.")
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map={"": 0},
        dtype=compute_dtype, attn_implementation="sdpa",
    )
    model.config.use_cache = False
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    print("Loaded in 4-bit:", model.is_loaded_in_4bit, "| Compute dtype:", compute_dtype)
else:
    print("Data checks complete. Set RUN_TRAINING=True and rerun from Configuration to train.")

## Attach LoRA adapters

- Freeze base weights without casting all floating-point weights to FP32.
- Adapt the existing attention/MLP projection selection in the text backbone.
- Non-reentrant gradient checkpointing is enabled by the trainer.

In [ ]:
if RUN_TRAINING:
    for param in model.parameters():
        param.requires_grad_(False)
    candidate_suffixes = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    target_modules = [
        name for name, module in model.named_modules()
        if name.startswith("model.layers.") and isinstance(module, torch.nn.Linear)
        and name.rsplit(".", 1)[-1] in candidate_suffixes
    ]
    if not target_modules:
        raise RuntimeError("No text projection targets found; inspect model.named_modules().")
    print("LoRA projection layers:", len(target_modules))
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", task_type="CAUSAL_LM", target_modules=target_modules,
    ))
    model.print_trainable_parameters()

## Shared grading helpers

- Define generation and grading metrics before training so each checkpoint can be assessed on validation answers.
- Generate from the question, desired answer, student answer, and criteria; ground-truth completions are never supplied.
- Use the same helpers later for test evaluation, including invalid and recovered-output reporting.


In [ ]:
def parse_prediction(raw_output, expected_count):
    recovered = False
    try:
        scores = json.loads(raw_output)
    except json.JSONDecodeError:
        candidates = re.findall(r"\[\s*[01](?:\s*,\s*[01])*\s*\]", raw_output)
        scores = next((
            parsed for candidate in reversed(candidates)
            if len(parsed := json.loads(candidate)) == expected_count
        ), None)
        recovered = scores is not None
    error = None
    if not isinstance(scores, list):
        error = "Output was not a JSON array."
    elif len(scores) != expected_count:
        error = f"Expected {expected_count} scores, received {len(scores)}."
    elif any(type(score) is not int or score not in (0, 1) for score in scores):
        error = "Scores must be integer 0 or 1."
    return {
        "valid": error is None, "raw_output": raw_output,
        "scores": scores if error is None else None,
        "error": error, "recovered": recovered,
    }


@torch.inference_mode()
def predict_criteria_scores(question, desired_answer, student_answer, criteria, prediction_model=None):
    prediction_model = model if prediction_model is None else prediction_model
    inputs = tokenizer.apply_chat_template(
        make_prompt(question, desired_answer, student_answer, criteria),
        tokenize=True, return_dict=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=False,
    ).to(prediction_model.get_input_embeddings().weight.device)
    generated = prediction_model.generate(
        **inputs, do_sample=False, use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        max_new_tokens=max(GENERATION_MIN_TOKENS, len(criteria) * GENERATION_TOKENS_PER_CRITERION),
    )
    answer_ids = generated[0, inputs["input_ids"].shape[-1]:]
    raw_output = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
    return parse_prediction(raw_output, len(criteria))

In [ ]:
def evaluate_current_model(dataset, model_name, prediction_model=None, show_progress=False):
    """Generate predictions and calculate grading metrics for one model state."""
    criterion_true = []
    criterion_pred = []
    exact_correct = 0
    invalid_count = 0
    final_score_errors = []
    final_score_exact = 0
    final_score_within_one = 0
    valid_true_final_scores = []
    valid_predicted_final_scores = []
    rows = []

    for example in tqdm(dataset, desc=f"{model_name} grades", disable=not show_progress):
        true_scores = example["criteria_score"]
        max_scores = example["criteria_max_score"]
        true_final_score = example["student_score"]
        prediction = predict_criteria_scores(
            example["question"], example["desired_answer"],
            example["student_answer"], example["criteria"], prediction_model=prediction_model,
        )

        # Invalid arrays are failures for response-level exact matching.
        is_valid = prediction["valid"]
        pred_scores = prediction["scores"] if is_valid else None
        is_exact = bool(is_valid and pred_scores == true_scores)
        exact_correct += int(is_exact)
        invalid_count += int(not is_valid)

        # Criterion metrics require aligned arrays, so invalid arrays are omitted.
        if is_valid:
            criterion_true.extend(true_scores)
            criterion_pred.extend(pred_scores)

            # Convert binary criterion decisions into the final numeric score.
            predicted_final_score = sum(
                score * maximum for score, maximum in zip(pred_scores, max_scores)
            )
            score_error = abs(predicted_final_score - true_final_score)
            final_score_errors.append(score_error)
            final_score_exact += int(score_error == 0)
            final_score_within_one += int(score_error <= 1)
            valid_true_final_scores.append(true_final_score)
            valid_predicted_final_scores.append(predicted_final_score)
        else:
            predicted_final_score = None
            score_error = None

        rows.append({
            "model": model_name,
            "row_index": example["row_index"],
            "raw_output": prediction["raw_output"],
            "recovered": prediction["recovered"],
            "error": prediction["error"],
            "ground_truth": true_scores,
            "prediction": pred_scores if is_valid else prediction["raw_output"],
            "valid": is_valid,
            "exact_match": is_exact,
            "true_final_score": true_final_score,
            "predicted_final_score": predicted_final_score,
            "final_score_absolute_error": score_error,
        })

    total = len(dataset)
    precision, recall, f1, _ = (
        precision_recall_fscore_support(criterion_true, criterion_pred, average="binary", zero_division=0)
        if criterion_true else (float("nan"),) * 4
    )
    if len(valid_true_final_scores) > 1 and len(set(valid_true_final_scores)) > 1 and len(set(valid_predicted_final_scores)) > 1:
        final_score_pearson = float(np.corrcoef(valid_true_final_scores, valid_predicted_final_scores)[0, 1])
    else:
        final_score_pearson = float("nan")
    if valid_true_final_scores:
        final_score_qwk = float(cohen_kappa_score(
            valid_true_final_scores, valid_predicted_final_scores, weights="quadratic"
        ))
    else:
        final_score_qwk = float("nan")

    metrics = {
        "model": model_name,
        "examples": total,
        "valid_examples": total - invalid_count,
        "recovered_output_rate": sum(row["recovered"] for row in rows) / total if total else 0.0,
        "invalid_output_rate": invalid_count / total if total else 0.0,
        "exact_match_accuracy": exact_correct / total if total else 0.0,
        "per_criterion_accuracy": accuracy_score(criterion_true, criterion_pred) if criterion_true else float("nan"),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        # Invalid generations count as incorrect for the two score accuracies.
        "final_score_exact_accuracy": final_score_exact / total if total else 0.0,
        "final_score_within_one_accuracy": final_score_within_one / total if total else 0.0,
        # MAE, RMSE, Pearson, and QWK describe valid numeric predictions only.
        "final_score_mae": (
            sum(final_score_errors) / len(final_score_errors)
            if final_score_errors else float("nan")
        ),
        "final_score_rmse": (
            (sum(error ** 2 for error in final_score_errors) / len(final_score_errors)) ** 0.5
            if final_score_errors else float("nan")
        ),
        "final_score_pearson": final_score_pearson,
        "final_score_qwk": final_score_qwk,
    }
    return metrics, rows


## Training configuration

- Paged 8-bit AdamW, mixed precision, and non-reentrant gradient checkpointing.
- Standard NLL with explicit completion labels; dynamic padding and no packing.
- Evaluate/save approximately every quarter epoch, rounded to an optimizer step.
- Generate validation grades at every check; invalid arrays count as incorrect for exact accuracy.
- Plateau scheduler maximizes the selected grading accuracy; patience 1 reduces LR after two unimproved checks.
- Early stopping uses the same grading metric; restore the highest-accuracy checkpoint.
- Validation loss is still logged but does not select checkpoints.


In [ ]:
microbatches_per_epoch = math.ceil(len(formatted_train) / TRAIN_BATCH_SIZE)
optimizer_steps_per_epoch = math.ceil(microbatches_per_epoch / GRADIENT_ACCUMULATION_STEPS)
quarter_epoch_steps = max(1, round(optimizer_steps_per_epoch / VALIDATIONS_PER_EPOCH))
print("Optimizer steps per epoch:", optimizer_steps_per_epoch)
print("Evaluate/save every:", quarter_epoch_steps, "optimizer steps")

if RUN_TRAINING:
    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        seed=SEED,
        data_seed=SEED,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        optim="paged_adamw_8bit",
        lr_scheduler_type="reduce_lr_on_plateau",
        lr_scheduler_kwargs={
            "mode": "max", "factor": LR_REDUCTION_FACTOR,
            "patience": LR_PATIENCE, "min_lr": MIN_LEARNING_RATE,
        },
        bf16=(compute_dtype == torch.bfloat16),
        fp16=(compute_dtype == torch.float16),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        activation_offloading=False,
        loss_type="nll",
        completion_only_loss=True,
        max_length=sequence_limit,
        packing=False,
        padding_free=False,
        logging_steps=max(1, quarter_epoch_steps // 2),
        save_strategy="steps",
        save_steps=quarter_epoch_steps,
        eval_strategy="steps",
        eval_steps=quarter_epoch_steps,
        prediction_loss_only=True,
        load_best_model_at_end=True,
        metric_for_best_model=CHECKPOINT_METRIC,
        greater_is_better=True,
        save_total_limit=3,
        report_to="none",
    )

## Create the trainer

- Add generated validation metrics inside the evaluation loop, before logging and early-stopping callbacks.
- The trainer receives only the structured validation split for grading; test generation happens after training.
- Training mode is restored even if validation generation fails; generation enables its own KV cache.


In [ ]:
class GradingMetricSFTTrainer(SFTTrainer):
    """Include free-running validation grades before Trainer selects checkpoints."""

    def __init__(self, *args, grading_validation_dataset, **kwargs):
        if len(grading_validation_dataset) == 0:
            raise ValueError("Grading checkpoint selection requires a nonempty validation set.")
        self.grading_validation_dataset = grading_validation_dataset
        super().__init__(*args, **kwargs)
        if self.args.world_size != 1:
            raise ValueError("Generated grading validation currently supports one training GPU.")

    def evaluation_loop(
        self, dataloader, description, prediction_loss_only=None,
        ignore_keys=None, metric_key_prefix="eval",
    ):
        was_training = self.model.training
        try:
            output = super().evaluation_loop(
                dataloader, description, prediction_loss_only=prediction_loss_only,
                ignore_keys=ignore_keys, metric_key_prefix=metric_key_prefix,
            )
            # Trainer.predict() must not attach validation grades to another dataset.
            if metric_key_prefix != "eval":
                return output
            self.model.eval()
            with self.autocast_smart_context_manager():
                grading_metrics, _ = evaluate_current_model(
                    self.grading_validation_dataset, "Validation",
                    prediction_model=self.model, show_progress=True,
                )
            output.metrics.update({
                f"{metric_key_prefix}_grading_{key}": value
                for key, value in grading_metrics.items() if key != "model"
            })
            selected_metric = output.metrics[self.args.metric_for_best_model]
            if not math.isfinite(selected_metric):
                raise ValueError("The checkpoint grading metric must be finite.")
            return output
        finally:
            self.model.train(was_training)


if RUN_TRAINING:
    trainer = GradingMetricSFTTrainer(
        model=model, args=training_args,
        train_dataset=formatted_train, eval_dataset=formatted_validation,
        # Remove assistant completions so generation uses prompt fields only.
        grading_validation_dataset=validation_dataset.remove_columns(["completion"]),
        processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )],
    )


## Check the collated loss mask

- Inspect a padded batch as seen by the trainer.
- Prompt and padding labels must be `-100`; completion tokens must remain supervised.

In [ ]:
if RUN_TRAINING:
    samples = [trainer.train_dataset[i] for i in range(min(2, len(trainer.train_dataset)))]
    batch = trainer.data_collator(samples)
    for index, sample in enumerate(samples):
        labels = batch["labels"][index]
        length = len(sample["labels"])
        assert labels[:length].tolist() == sample["labels"], "Collator changed the completion mask."
        assert (labels[length:] == -100).all(), "Padding contributes to loss."
        assert (labels[:length] == -100).any() and (labels[:length] != -100).any()
    print("Prompt/completion/padding masks verified.")

## Train and save

- Only LoRA parameters are updated.
- Save the best adapter and tokenizer after training succeeds.
- Print and save the selected checkpoint/validation accuracy and per-check validation metrics for comparison.


In [ ]:
if RUN_TRAINING:
    train_result = trainer.train()
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    checkpoint_selection = {
        "metric": CHECKPOINT_METRIC,
        "greater_is_better": True,
        "best_validation_metric": trainer.state.best_metric,
        "best_model_checkpoint": trainer.state.best_model_checkpoint,
        "split_seed": SPLIT_SEED,
        "validation_examples": len(validation_dataset),
        "validation_questions": len(question_splits["validation"]),
    }
    Path(OUTPUT_DIR, "checkpoint_selection.json").write_text(
        json.dumps(checkpoint_selection, indent=2) + "\n"
    )
    validation_history = [
        entry for entry in trainer.state.log_history if CHECKPOINT_METRIC in entry
    ]
    pd.DataFrame(validation_history).to_csv(
        Path(OUTPUT_DIR, "validation_grading_metrics.csv"), index=False,
    )
    print("Saved best adapter and tokenizer to:", OUTPUT_DIR)
    print("Selected checkpoint:", trainer.state.best_model_checkpoint)
    print(f"Best {CHECKPOINT_METRIC}: {trainer.state.best_metric}")
    display(pd.DataFrame(validation_history))
    display(train_result.metrics)


## Prepare inference

- Reuse the trained model and enable its KV cache.
- Release optimizer/gradient storage before generation.

In [ ]:
if RUN_TRAINING:
    model = trainer.model
    model.zero_grad(set_to_none=True)
    trainer.optimizer = None
    trainer.lr_scheduler = None
    # Accelerator also retains references to optimizer wrappers.
    trainer.accelerator.free_memory()
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    model.eval()

## Manual prediction

- Expected rubric array: `[1, 0, 0]`.
- Runs only after training is enabled.

In [ ]:
if RUN_TRAINING:
    result = predict_criteria_scores(
        question="Explain why HTTPS is more secure than HTTP.",
        desired_answer="HTTPS encrypts network traffic using TLS.",
        student_answer="HTTPS protects transmitted data using encryption.",
        criteria=["Mentions encryption", "Mentions TLS", "Claims HTTPS sends plaintext"],
    )
    print(result)

## Evaluate the held-out test set

- Use one evaluator for the trained adapter and disabled-adapter baseline.
- Invalid outputs count as failures for exact-array and final-score accuracy.
- Per-criterion accuracy/precision/recall/F1 and score MAE/RMSE use valid outputs only.
- Final score = sum of predicted criterion decision × criterion maximum.
- Report invalid and recovered-output rates alongside metrics.

In [ ]:
if RUN_TRAINING and RUN_TEST_EVALUATION:
    finetuned_metrics, finetuned_rows = evaluate_current_model(test_dataset, "Fine-tuned QLoRA")
    display(pd.Series(finetuned_metrics))

## Inspect predictions

- Keep raw output, validity, recovered status, criterion arrays, and numeric score errors.

In [ ]:
if RUN_TRAINING and RUN_TEST_EVALUATION:
    evaluation_df = pd.DataFrame(finetuned_rows)
    display(evaluation_df.head(20))

## Baseline comparison

- Temporarily disable LoRA; reuse the same model, test set, prompt, parser, and decoding.
- Reuse fine-tuned metrics; generate only the baseline pass here.
- The delta row is fine-tuned minus baseline: positive is better for accuracy/F1, negative for errors.
- Adapter disabling restores the trained adapter when the context exits.

In [ ]:
if RUN_TRAINING and RUN_TEST_EVALUATION:
    with model.disable_adapter():
        baseline_metrics, baseline_rows = evaluate_current_model(test_dataset, "Baseline Qwen3.5-9B")
    comparison_df = pd.DataFrame([baseline_metrics, finetuned_metrics]).set_index("model")
    comparison_df.loc["Delta (fine-tuned - baseline)"] = (
        comparison_df.loc["Fine-tuned QLoRA"] - comparison_df.loc["Baseline Qwen3.5-9B"]
    )
    final_score_predictions_df = pd.DataFrame(baseline_rows + finetuned_rows)
    display(comparison_df)
    display(final_score_predictions_df[[
        "model", "row_index", "true_final_score", "predicted_final_score",
        "final_score_absolute_error", "valid",
    ]].head(20))
    evaluation_completed = True

## Upload the evaluated adapter to Hugging Face

- Set `RUN_HF_UPLOAD=True` in Configuration to opt in.
- Upload runs only after both training and held-out test evaluation are enabled and complete.
- `OUTPUT_DIR` contains the final LoRA adapter and tokenizer; intermediate `checkpoint-*` folders are excluded.
- `login()` prompts for a Hugging Face write token. The target model repository must already exist and your account must be allowed to write to it.

In [ ]:
if RUN_HF_UPLOAD:
    if not (RUN_TRAINING and RUN_TEST_EVALUATION):
        raise RuntimeError(
            "Set RUN_TRAINING=True and RUN_TEST_EVALUATION=True before uploading."
        )
    if not evaluation_completed:
        raise RuntimeError("Complete the held-out evaluation cells before uploading.")
    login()
    upload_result = upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HF_REPO_ID,
        repo_type="model",
        ignore_patterns=["checkpoint-*"],
        commit_message="Upload evaluated Qwen3.5-9B QLoRA adapter",
    )
    print("Uploaded evaluated adapter to:", upload_result)
else:
    print("Hugging Face upload skipped. Set RUN_HF_UPLOAD=True to publish.")